# Command Primitive

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn how to use `Command(update=, goto=, graph=, resume=)` to combine state updates and routing in a single return value, return Command from nodes and tools, and navigate subgraphs.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Command: Update + Goto

A node returns `Command` to set state and choose the next node in one step.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Command

class State(TypedDict):
    messages: Annotated[list, add_messages]
    category: str

def classifier(state: State) -> Command[Literal["poem", "joke"]]:
    last_msg = state["messages"][-1].content.lower()
    if "poem" in last_msg:
        return Command(update={"category": "poem"}, goto="poem")
    return Command(update={"category": "joke"}, goto="joke")

def poem_node(state: State) -> dict:
    return {"messages": [("ai", "Roses are red, violets are blue...")]}

def joke_node(state: State) -> dict:
    return {"messages": [("ai", "Why did the chicken cross the road?")]}

graph = StateGraph(State)
graph.add_node("classifier", classifier)
graph.add_node("poem", poem_node)
graph.add_node("joke", joke_node)
graph.add_edge(START, "classifier")
graph.add_edge("poem", END)
graph.add_edge("joke", END)

app = graph.compile()

result = app.invoke({"messages": [("human", "Write me a poem about spring")]})
print(f"Category: {result['category']}")
print(f"Output: {result['messages'][-1].content}")

In [ ]:
result = app.invoke({"messages": [("human", "Tell me a joke")]})
print(f"Category: {result['category']}")
print(f"Output: {result['messages'][-1].content}")

## 4. Command with LLM Routing

Use an LLM to decide the routing while updating state via Command.

In [ ]:
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

class RouteDecision(BaseModel):
    category: Literal["poem", "joke"]

llm = ChatOpenAI(model="gpt-4o-mini")
router_llm = llm.with_structured_output(RouteDecision)

def llm_classifier(state: State) -> Command[Literal["poem", "joke"]]:
    result = router_llm.invoke(state["messages"])
    return Command(update={"category": result.category}, goto=result.category)

def poem_gen(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a poet. Write a short poem."),
        *state["messages"],
    ])
    return {"messages": [response]}

def joke_gen(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a comedian. Write a short joke."),
        *state["messages"],
    ])
    return {"messages": [response]}

graph2 = StateGraph(State)
graph2.add_node("classifier", llm_classifier)
graph2.add_node("poem", poem_gen)
graph2.add_node("joke", joke_gen)
graph2.add_edge(START, "classifier")
graph2.add_edge("poem", END)
graph2.add_edge("joke", END)

app2 = graph2.compile()

result = app2.invoke({"messages": [("human", "Make me laugh about programming")]})
print(f"Category: {result['category']}")
print(f"Output:\n{result['messages'][-1].content}")

## 5. Command from Tools

Tools can return `Command` to control graph flow when called by an LLM agent.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny and 72°F."

@tool
def escalate(reason: str) -> Command:
    """Escalate to a human agent when you cannot help."""
    return Command(
        update={"messages": [("ai", f"Escalating to human support: {reason}")]},
        goto="human_support",
    )

print(f"Tools defined: {get_weather.name}, {escalate.name}")

## 6. Fan-Out with goto List

`goto` accepts a list of node names to execute multiple nodes in parallel.

In [ ]:
import operator

class FanOutState(TypedDict):
    input: str
    results: Annotated[list[str], operator.add]

def dispatch(state: FanOutState) -> Command[Literal["analyzer", "summarizer"]]:
    return Command(
        update={"results": ["dispatched"]},
        goto=["analyzer", "summarizer"],
    )

def analyzer(state: FanOutState) -> dict:
    return {"results": [f"analyzed: {state['input']}"]}

def summarizer(state: FanOutState) -> dict:
    return {"results": [f"summarized: {state['input']}"]}

graph3 = StateGraph(FanOutState)
graph3.add_node("dispatch", dispatch)
graph3.add_node("analyzer", analyzer)
graph3.add_node("summarizer", summarizer)
graph3.add_edge(START, "dispatch")
graph3.add_edge("analyzer", END)
graph3.add_edge("summarizer", END)

app3 = graph3.compile()
result = app3.invoke({"input": "LangGraph is great", "results": []})
print(f"Results: {result['results']}")

## What You Learned

1. **`Command(update=, goto=)`** combines state updates and routing in one return value
2. Type-hint returns as **`Command[Literal[...]]`** for graph validation
3. **Tools** can return `Command` to control graph flow
4. **`goto` accepts a list** of node names for parallel fan-out
5. **`graph=Command.PARENT`** enables subgraph-to-parent navigation